# Business Analysis
## Last-Mile Delivery Operations

Moves beyond descriptive EDA into statistical testing, segment ranking,
and root-cause (interaction) analysis. Every finding below follows
Finding -> Evidence -> Business Impact so it's ready to feed directly
into Insights & Recommendations.

In [2]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from config.config import PROCESSED_DATA_DIR
from src.analysis.business_analysis import (
    run_significance_scan, rank_segments_by_score,
    weather_mode_interaction, identify_high_risk_segment,
    cost_efficiency_by_segment
)

df = pd.read_csv(PROCESSED_DATA_DIR / "delivery_logistics_features.csv")
df.shape

(25000, 24)

## 1. Which segments actually matter? (Statistical Significance Scan)

EDA suggested weather and delivery_mode look like strong
drivers, while region/partner/vehicle/package_type looked flat. This
formally tests that impression with a chi-square test of independence
against `delivery_status` for every categorical segment.

In [3]:
columns_to_test = ['region', 'delivery_partner', 'vehicle_type',
                    'weather_condition', 'delivery_mode', 'package_type']

significance_results = run_significance_scan(df, columns_to_test)
significance_results

,column,chi2,p_value,dof,significant_at_0.05
0,delivery_mode,11654.49,0.000000,6,True
1,weather_condition,1359.97,0.000000,10,True
2,delivery_partner,26.80,0.043813,16,True
3,package_type,14.55,0.557493,16,False
4,region,8.31,0.403729,8,False
5,vehicle_type,5.32,0.868849,10,False


**Finding:** Only two segments show a statistically meaningful
association with delivery outcome:

- **`delivery_mode`**: chi2 = 11,654.5, p < 0.0001 -- an extremely
  strong, unambiguous effect.
- **`weather_condition`**: chi2 = 1,360.0, p < 0.0001 -- also strongly
  significant.

`region` (p = 0.40) and `vehicle_type` (p = 0.87) and `package_type`
(p = 0.56) show no statistically significant association at all.
`delivery_partner` is technically significant (p = 0.044) but its chi2
(26.8) is tiny compared to weather/mode, and the earlier EDA rate
differences between partners were only 1-2 percentage points --
statistically detectable due to the large sample size, but not
practically meaningful for operational decisions.

**Business Impact:** Management effort spent trying to differentiate
regions, vehicle types, or partners on delivery outcome would largely
be wasted. The two things actually worth investigating and acting on
are **weather-driven risk** and **delivery mode reliability**.

## 2. Root Cause: Does Weather Affect All Delivery Modes Equally?

Weather and delivery_mode are both significant on their own -- but do
they interact? I.e. is weather's damage concentrated in one mode, or
spread evenly?

In [4]:
interaction = weather_mode_interaction(df, modes=['express', 'same day'])
interaction

delivery_status                  delayed  delivered  failed
delivery_mode weather_condition                            
express       clear                 42.5       48.4     9.1
              cold                  43.0       47.1     9.9
              foggy                 71.7       10.7    17.7
              hot                   41.6       48.0    10.4
              rainy                 78.5        2.3    19.2
              stormy                79.4        0.2    20.4
same day      clear                 13.2       83.5     3.3
              cold                   9.0       88.2     2.8
              foggy                 28.2       65.2     6.6
              hot                   12.8       84.9     2.3
              rainy                 41.8       47.6    10.6
              stormy                50.5       34.4    15.1

**Finding -- this is the central root cause of the project.** Weather's
damage is heavily concentrated in `express` mode, not spread evenly:

| Mode | Weather | Delayed % | Failed % | Delivered % |
|---|---|---|---|---|
| Express | Clear | 42.5% | 9.1% | 48.4% |
| Express | Stormy | 79.4% | 20.4% | **0.2%** |
| Same day | Clear | 13.2% | 3.3% | 83.5% |
| Same day | Stormy | 50.5% | 15.1% | 34.4% |

Even in **clear weather**, express mode only delivers on time 48.4% of
the time -- it is fundamentally over-promised relative to what the
operation can reliably achieve. Under stormy conditions, express
delivery essentially collapses: only 0.2% arrive on time. Same day mode
degrades under bad weather too, but far less severely (83.5% -> 34.4%
delivered) than express (48.4% -> 0.2%).

**Business Impact:** The express delivery promise is not primarily a
weather problem -- it's an underlying reliability problem that weather
makes catastrophic. Fixing weather-related delays alone (e.g. buffer
time) would help, but the core issue is that express-mode SLAs appear
to be set without enough margin even in good conditions.

## 3. Quantifying the Highest-Risk Operational Segment

Combining the two significant drivers: express deliveries during
rainy or stormy weather.

In [6]:
high_risk = identify_high_risk_segment(df, weather_values=['rainy', 'stormy'], mode_value='express')
high_risk

{'segment_size': 2077,
 'pct_of_total_volume': 8.3,
 'segment_status_distribution': {'delayed': 79.0,
  'failed': 19.8,
  'delivered': 1.3},
 'overall_baseline_distribution': {'delivered': 73.3,
  'delayed': 21.4,
  'failed': 5.3}}

**Finding:** This single segment (express + rainy/stormy) makes up
8.3% of all deliveries (2,077 of 25,000) but has a delivered rate of
just **1.3%**, versus 73.3% overall -- delayed jumps to 79.0% and failed
to 19.8%. This is a small share of volume causing a disproportionate
share of customer harm and likely a disproportionate share of failed-
delivery cost (redelivery, refunds, support tickets).

**Business Impact:** This is the single most actionable, well-defined
segment in the dataset. It's specific enough to act on directly (e.g.
suspend or de-prioritize express promises when weather forecasts show
rain/storms) rather than a vague "improve reliability" directive.

## 4. Is the `standard` Mode's Perfect Record a Data Artifact?

EDA flagged that `standard` mode shows a 100% delivered rate with zero
delays or failures -- unusually clean. Checking whether this is
explained by a hidden bias (e.g. only short-distance deliveries).

In [7]:
standard_df = df[df['delivery_mode'] == 'standard']
express_df = df[df['delivery_mode'] == 'express']

print("standard mode -- distance_km stats:")
print(standard_df['distance_km'].describe())
print()
print("express mode -- distance_km stats:")
print(express_df['distance_km'].describe())

standard mode -- distance_km stats:
count    6186.000000
mean      148.610734
std        85.760032
min         3.600000
25%        74.800000
50%       147.000000
75%       222.500000
max       297.100000
Name: distance_km, dtype: float64

express mode -- distance_km stats:
count    6233.000000
mean      150.954805
std        86.412012
min         3.600000
25%        76.100000
50%       151.700000
75%       225.100000
max       297.100000
Name: distance_km, dtype: float64


**Finding:** `standard` mode's average distance (148.6km) is nearly
identical to `express` mode's (151.0km) -- so the perfect record is
**not** explained by standard mode secretly only covering short,
easy routes. Combined with the chi-square result above, this looks
like a genuine, deliberate operational segment: standard/two-day modes
appear to be built with wide enough time buffers to always succeed,
while express is built with almost no buffer at all.

**Business Impact:** The company already has proof, in its own data,
that reliable delivery is achievable regardless of distance -- the gap
is specifically in how the express promise is designed, not a
structural limitation of the delivery network.

## 5. Segment Ranking (for completeness, given weak significance)

Even though region/vehicle/partner showed weak or no statistical
significance, ranking them is still useful to confirm there's no single
underperforming outlier hiding inside an average.

In [8]:
print("=== Delivery Partners ===")
print(rank_segments_by_score(df, 'delivery_partner'))
print()
print("=== Vehicle Types ===")
print(rank_segments_by_score(df, 'vehicle_type'))
print()
print("=== Regions ===")
print(rank_segments_by_score(df, 'region'))

=== Delivery Partners ===
                  avg_performance_score  delivery_count
delivery_partner                                       
delhivery                         80.71            2786
fedex                             80.70            2818
ecom express                      80.07            2722
dhl                               79.80            2802
shadowfax                         79.59            2736
blue dart                         79.55            2798
ekart                             79.32            2801
amazon logistics                  79.28            2711
xpressbees                        78.57            2826

=== Vehicle Types ===
              avg_performance_score  delivery_count
vehicle_type                                       
scooter                       80.17            4174
ev bike                       79.90            4218
ev van                        79.85            4116
van                           79.64            4187
truck                  

**Finding:** No outliers found. Every partner scores within 78.6-80.7
(out of 100), every vehicle type within 79.4-80.2, every region within
79.4-80.3. No single partner, vehicle, or region is dragging down the
average -- confirming the chi-square results weren't hiding one bad
performer inside a "not significant overall" result.

**Business Impact:** Confirms there's no quick win from switching a
specific underperforming partner or reallocating a specific region --
the real opportunity is structural (mode design + weather
contingency), not a personnel/vendor swap.

## 6. Cost Efficiency Check

Testing whether any vehicle type or partner is meaningfully less cost-
efficient per kilometer, restricted to deliveries over 50km to avoid
the fixed-cost skew on short trips.

In [7]:
print("=== Cost per km by vehicle type (>50km) ===")
print(cost_efficiency_by_segment(df, 'vehicle_type'))
print()
print("=== Cost per km by partner (>50km) ===")
print(cost_efficiency_by_segment(df, 'delivery_partner'))

=== Cost per km by vehicle type (>50km) ===
              avg_cost_per_km
vehicle_type                 
van                      5.84
ev bike                  5.81
truck                    5.81
scooter                  5.80
ev van                   5.80
bike                     5.79

=== Cost per km by partner (>50km) ===
                  avg_cost_per_km
delivery_partner                 
blue dart                    5.82
shadowfax                    5.82
delhivery                    5.82
fedex                        5.82
xpressbees                   5.81
ecom express                 5.81
dhl                          5.80
amazon logistics             5.79
ekart                        5.79


**Finding:** Cost per km is essentially flat across vehicle types
(₹5.79-5.84) and partners (₹5.79-5.82) once short-distance skew is
removed. There is no meaningful cost inefficiency tied to vehicle
choice or partner choice in this dataset.

**Business Impact:** Cost optimization efforts aimed at vehicle mix or
partner renegotiation are unlikely to move the needle much -- cost is
overwhelmingly explained by distance alone (r ≈ 0.99, from EDA), not by
who or what is doing the delivering.

## Business Analysis Summary

| Question | Verdict | Confidence |
|---|---|---|
| Do regions differ in performance? | No | Not statistically significant (p=0.40) |
| Do vehicle types differ? | No | Not statistically significant (p=0.87) |
| Does package type matter? | No | Not statistically significant (p=0.56) |
| Do partners differ? | Marginally, negligibly | Significant (p=0.04) but tiny effect size |
| Does weather matter? | **Yes, strongly** | p<0.0001, large effect |
| Does delivery mode matter? | **Yes, very strongly** | p<0.0001, largest effect in dataset |
| Root cause of weather's damage | Concentrated in express mode | Interaction analysis, Section 2 |
| Highest-risk segment | Express + rainy/stormy weather | 8.3% of volume, 1.3% delivered rate |
| Is 'standard' mode's perfect record real? | Yes, confirmed | Not explained by distance bias |
| Cost inefficiency by vehicle/partner? | None found | Flat ~₹5.79-5.84/km |

This sets up Insights & Recommendations with two clear,
evidence-backed priorities: **(1) redesign express-mode SLAs/buffers**
and **(2) build weather-contingent handling for express deliveries**,
rather than pursuing regional, partner, or vehicle-level interventions
the data doesn't support.